# MNIST Digit Classification with CNN (PyTorch)

A convolutional neural network trained on the MNIST dataset to classify handwritten digits (0–9).  
Reaches ~99% accuracy on the test set after 10 epochs.

**Pipeline:**
1. Load & inspect MNIST
2. Preprocess — tensor conversion, train/val/test split
3. Define CNN architecture
4. Train with Adam + LR scheduler
5. Evaluate with accuracy, precision, recall
6. Plot training curves
7. Save model weights

## 1. Imports & Reproducibility

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy, Precision, Recall

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2. Load MNIST

In [ ]:
# ToTensor() converts PIL images to float tensors and scales pixels from [0, 255] to [0.0, 1.0].
transform = transforms.Compose([transforms.ToTensor()])

train_data = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

print(f"Train: {len(train_data):,} | Test: {len(test_data):,}")

## 3. Visualise Samples

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
indices = np.random.randint(0, len(train_data), size=8)

for ax, idx in zip(axes, indices):
    image, label = train_data[idx]
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(str(label), fontsize=13)
    ax.axis("off")

plt.suptitle("Random samples from training set", y=1.02)
plt.tight_layout()
plt.show()

## 4. Split — Train / Validation / Test

In [ ]:
# Split the 10k test set 50/50 into validation and final test.
# Validation guides training decisions; test set is untouched until the very end.
cv_size   = len(test_data) // 2
test_size = len(test_data) - cv_size

generator = torch.Generator().manual_seed(SEED)
cv_data, test_data = random_split(test_data, [cv_size, test_size], generator=generator)

print(f"Train: {len(train_data):,} | Val: {len(cv_data):,} | Test: {len(test_data):,}")

## 5. Preprocess — Convert to Tensors

In [ ]:
def dataset_to_tensors(dataset):
    """
    Converts a PyTorch Dataset into (X, y) tensors.

    Returns:
        x : float tensor [N, 28, 28]  — normalised pixel values
        y : long tensor  [N]          — integer class labels (0-9)

    Labels stay as integers because CrossEntropyLoss expects class indices,
    not one-hot vectors. No conversion needed.
    """
    images, labels = zip(*[(img.squeeze(), lbl) for img, lbl in dataset])
    x = torch.stack(images)                     # [N, 28, 28]
    y = torch.tensor(labels, dtype=torch.long)  # [N]
    return x, y


train_x, train_y = dataset_to_tensors(train_data)
cv_x,    cv_y    = dataset_to_tensors(cv_data)
test_x,  test_y  = dataset_to_tensors(test_data)

print("Shapes:")
for name, x, y in [("train", train_x, train_y), ("val", cv_x, cv_y), ("test", test_x, test_y)]:
    print(f"  {name:5s}  x: {tuple(x.shape)}  y: {tuple(y.shape)}  dtype: {y.dtype}")

## 6. DataLoader

In [ ]:
TRAIN_SUBSET = 5_000   # increase to train_x.shape[0] for full training
BATCH_SIZE   = 32

train_dataset = TensorDataset(
    train_x[:TRAIN_SUBSET].to(DEVICE),
    train_y[:TRAIN_SUBSET].to(DEVICE),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

print(f"Batches per epoch: {len(train_loader)}  (batch size: {BATCH_SIZE})")

## 7. Model Architecture

In [ ]:
def build_model() -> nn.Sequential:
    """
    Small CNN for 10-class digit classification.

    Input  : [B, 1, 28, 28]
    Output : [B, 10]  raw logits

    Layer flow:
        ZeroPad2d(2)        [B,  1, 32, 32]  pad before conv to preserve spatial size
        Conv2d(1->16, k=5)  [B, 16, 28, 28]  16 filters learn local visual features
        BatchNorm2d(16)     [B, 16, 28, 28]  stabilise activations across the batch
        ReLU                [B, 16, 28, 28]  non-linearity
        MaxPool2d(2)        [B, 16, 14, 14]  halve spatial dims, add position tolerance
        Flatten             [B, 3136]
        LazyLinear(10)      [B, 10]          one score per class

    No Softmax: CrossEntropyLoss applies log-softmax internally.
    Adding it explicitly causes numerical instability.
    """
    return nn.Sequential(
        nn.ZeroPad2d(2),
        nn.Conv2d(in_channels=1, out_channels=16, kernel_size=5, stride=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
        nn.Flatten(),
        nn.LazyLinear(out_features=10),
    )


model = build_model().to(DEVICE)

try:
    from torchsummary import summary
    summary(model, input_size=(1, 28, 28))
except ImportError:
    print(model)

## 8. Loss, Optimiser, Scheduler

In [ ]:
# CrossEntropyLoss expects raw logits + integer class labels [N].
criterion = nn.CrossEntropyLoss()

# Adam default lr is 1e-3. Higher values overshoot minima on harder datasets.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Linearly decay lr from 1e-3 to 1e-4 over 10 epochs.
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1.0, end_factor=0.1, total_iters=10
)

## 9. Metrics

In [ ]:
NUM_CLASSES = 10

def make_metrics(device):
    # Metrics must live on the same device as the model to avoid
    # device mismatch errors when running on GPU.
    return {
        "accuracy" : Accuracy(num_classes=NUM_CLASSES,  task="multiclass").to(device),
        "precision": Precision(num_classes=NUM_CLASSES, task="multiclass").to(device),
        "recall"   : Recall(num_classes=NUM_CLASSES,    task="multiclass").to(device),
    }

train_metrics = make_metrics(DEVICE)
cv_metrics    = make_metrics(DEVICE)

## 10. Training & Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, metrics):
    model.train()
    total_loss = 0.0

    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        # Conv2d expects [B, C, H, W]; data is [B, H, W] so we insert the channel dim.
        logits = model(inputs.unsqueeze(1))   # [B, 10] — no .squeeze() on output
        preds  = torch.argmax(logits, dim=1)  # [B] predicted class index

        # targets are integer labels [B] — exactly what CrossEntropyLoss expects.
        loss = criterion(logits, targets)
        total_loss += loss.item()

        for m in metrics.values():
            m.update(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return total_loss / len(loader)


def evaluate(model, x, y, metrics):
    model.eval()
    with torch.inference_mode():
        logits = model(x.unsqueeze(1))        # [N, 10]
        preds  = torch.argmax(logits, dim=1)  # [N]
        loss   = criterion(logits, y)
        for m in metrics.values():
            m.update(preds, y)
    return loss.item()

## 11. Training Loop

In [ ]:
NUM_EPOCHS = 10

cv_x_dev = cv_x[:900].to(DEVICE)
cv_y_dev = cv_y[:900].to(DEVICE)

history = {"train_loss": [], "cv_loss": [], "train_acc": [], "cv_acc": []}

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Val Loss':>8}  {'Train Acc':>9}  {'Val Acc':>7}  {'LR':>8}")
print("-" * 62)

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, train_metrics)
    cv_loss    = evaluate(model, cv_x_dev, cv_y_dev, cv_metrics)

    t_acc = train_metrics["accuracy"].compute().item()
    c_acc = cv_metrics["accuracy"].compute().item()
    lr    = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["cv_loss"].append(cv_loss)
    history["train_acc"].append(t_acc)
    history["cv_acc"].append(c_acc)

    print(f"{epoch:>5}  {train_loss:>10.4f}  {cv_loss:>8.4f}  {t_acc:>9.4f}  {c_acc:>7.4f}  {lr:>8.6f}")

    for m in list(train_metrics.values()) + list(cv_metrics.values()):
        m.reset()

    scheduler.step()

print("\nDone.")

## 12. Training Curves

In [ ]:
epochs = range(NUM_EPOCHS)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, history["train_loss"], label="Train", marker="o", linewidth=2)
ax1.plot(epochs, history["cv_loss"],    label="Val",   marker="o", linewidth=2, linestyle="--")
ax1.set_title("Loss per Epoch")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("CrossEntropy Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history["train_acc"], label="Train", marker="o", linewidth=2)
ax2.plot(epochs, history["cv_acc"],    label="Val",   marker="o", linewidth=2, linestyle="--")
ax2.set_title("Accuracy per Epoch")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_ylim(0.8, 1.01)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Training Curves — MNIST CNN", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")

## 13. Final Test Evaluation

In [ ]:
test_metrics = make_metrics(DEVICE)
evaluate(model, test_x.to(DEVICE), test_y.to(DEVICE), test_metrics)

print("Test results:")
for name, m in test_metrics.items():
    print(f"  {name.capitalize():10s}: {m.compute().item():.4f}")

## 14. Save Model Weights

In [ ]:
# Save state_dict only — portable across Python/PyTorch versions.
# Load with: model.load_state_dict(torch.load('mnist_cnn.pth'))
SAVE_PATH = "mnist_cnn.pth"
torch.save(model.state_dict(), SAVE_PATH)
print(f"Weights saved to '{SAVE_PATH}'")